In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("merged_fraud_dataset.csv",low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (57661, 45)


,transaction_id,transaction_date,customer_id,card_type,amount,merchant_category,merchant_country,transaction_channel,device_type,is_international,...,email_age_days,ip_risk_level,failed_payment_attempts,delivery_speed,customer_balance,location,distance_from_home_km,transactions_last_1h,new_device,new_location
0,CC00011340,2025-07-15 19:54:24,C002995,visa,55.82,travel,uae,website,windows,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CC00007384,2025-07-08 14:57:48,C003084,mastercard,85.33,travel,uae,atm,windows,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CC00002213,2025-05-02 20:09:53,C003082,amex,76.06,entertainment,united states,website,other,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CC00009658,2025-01-12 15:39:05,C002467,rupay,72.56,travel,united states,website,windows,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CC00009328,2025-07-31 05:04:38,C001852,rupay,257.07,fuel,uae,pos,android,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

numeric_columns = [
    "amount",
    "risk_score",
    "transactions_last_24h",
    "transactions_last_1h",
    "login_attempts",
    "failed_payment_attempts",
    "distance_from_home_km",
    "average_amount_30d",
    "account_age_days"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

print("Data types converted successfully.")

Data types converted successfully.


In [3]:
df["transaction_year"] = df["transaction_date"].dt.year
df["transaction_month"] = df["transaction_date"].dt.month
df["transaction_day"] = df["transaction_date"].dt.day
df["day_of_week"] = df["transaction_date"].dt.dayofweek
df["day_name"] = df["transaction_date"].dt.day_name()

df["transaction_hour"] = pd.to_numeric(
    df["transaction_hour"],
    errors="coerce"
)

df["transaction_hour"] = df["transaction_hour"].fillna(
    df["transaction_date"].dt.hour
)

df[[
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "day_name",
    "transaction_hour"
]].head()

,transaction_date,transaction_year,transaction_month,day_name,transaction_hour
0,2025-07-15 19:54:24,2025,7,Tuesday,19
1,2025-07-08 14:57:48,2025,7,Tuesday,14
2,2025-05-02 20:09:53,2025,5,Friday,20
3,2025-01-12 15:39:05,2025,1,Sunday,15
4,2025-07-31 05:04:38,2025,7,Thursday,5


In [4]:
df["is_weekend"] = df["day_of_week"].apply(
    lambda day: 1 if day >= 5 else 0
)

df[["day_name", "is_weekend"]].head()

,day_name,is_weekend
0,Tuesday,0
1,Tuesday,0
2,Friday,0
3,Sunday,1
4,Thursday,0


In [5]:
df["unusual_time_indicator"] = df["transaction_hour"].apply(
    lambda hour: 1 if pd.notna(hour) and 0 <= hour <= 5 else 0
)

df[[
    "transaction_hour",
    "unusual_time_indicator"
]].head()

,transaction_hour,unusual_time_indicator
0,19,0
1,14,0
2,20,0
3,15,0
4,5,1


In [6]:
amount_limit = df["amount"].quantile(0.90)

df["high_amount_indicator"] = df["amount"].apply(
    lambda amount: 1
    if pd.notna(amount) and amount > amount_limit
    else 0
)

print("High amount limit:", round(amount_limit, 2))

df[[
    "amount",
    "high_amount_indicator"
]].head()

High amount limit: 1142.05


,amount,high_amount_indicator
0,55.82,0
1,85.33,0
2,76.06,0
3,72.56,0
4,257.07,0


In [7]:
df["amount_group"] = pd.cut(
    df["amount"],
    bins=[0, 100, 500, 1000, 5000, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
        "Extremely High"
    ],
    include_lowest=True
)

df[["amount", "amount_group"]].head()

,amount,amount_group
0,55.82,Low
1,85.33,Low
2,76.06,Low
3,72.56,Low
4,257.07,Medium


In [8]:
df["amount_above_average_indicator"] = np.where(
    df["amount"].notna() &
    df["average_amount_30d"].notna() &
    (df["average_amount_30d"] > 0) &
    (df["amount"] > df["average_amount_30d"] * 2),
    1,
    0
)

df[[
    "amount",
    "average_amount_30d",
    "amount_above_average_indicator"
]].head()

,amount,average_amount_30d,amount_above_average_indicator
0,55.82,77.01,0
1,85.33,134.09,0
2,76.06,162.75,0
3,72.56,104.91,0
4,257.07,179.18,0


In [9]:
df["high_frequency_indicator"] = np.where(
    (
        df["transactions_last_24h"].fillna(0) >= 10
    ) |
    (
        df["transactions_last_1h"].fillna(0) >= 5
    ),
    1,
    0
)

df[[
    "transactions_last_24h",
    "transactions_last_1h",
    "high_frequency_indicator"
]].head()

,transactions_last_24h,transactions_last_1h,high_frequency_indicator
0,7.0,NaN,0
1,3.0,NaN,0
2,5.0,NaN,0
3,4.0,NaN,0
4,1.0,NaN,0


In [10]:
df["failed_attempt_indicator"] = np.where(
    (
        df["failed_payment_attempts"].fillna(0) >= 3
    ) |
    (
        df["login_attempts"].fillna(0) >= 5
    ),
    1,
    0
)

df[[
    "failed_payment_attempts",
    "login_attempts",
    "failed_attempt_indicator"
]].head()

,failed_payment_attempts,login_attempts,failed_attempt_indicator
0,NaN,NaN,0
1,NaN,NaN,0
2,NaN,NaN,0
3,NaN,NaN,0
4,NaN,NaN,0


In [11]:
df["far_from_home_indicator"] = np.where(
    df["distance_from_home_km"].fillna(0) > 100,
    1,
    0
)

df[[
    "distance_from_home_km",
    "far_from_home_indicator"
]].head()

,distance_from_home_km,far_from_home_indicator
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0


In [12]:
binary_columns = [
    "new_device",
    "new_location",
    "is_international",
    "is_card_present"
]

for column in binary_columns:
    df[column] = df[column].replace({
        "Yes": 1,
        "No": 0,
        "True": 1,
        "False": 0,
        True: 1,
        False: 0
    })

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df[binary_columns].head()

,new_device,new_location,is_international,is_card_present
0,NaN,NaN,0.0,1.0
1,NaN,NaN,0.0,0.0
2,NaN,NaN,1.0,1.0
3,NaN,NaN,0.0,0.0
4,NaN,NaN,1.0,0.0


In [13]:
df["new_device_indicator"] = np.where(
    df["new_device"] == 1,
    1,
    0
)

In [14]:
df["new_location_indicator"] = np.where(
    df["new_location"] == 1,
    1,
    0
)

In [15]:
df["international_indicator"] = np.where(
    df["is_international"] == 1,
    1,
    0
)

In [16]:
df["card_not_present_indicator"] = np.where(
    df["is_card_present"] == 0,
    1,
    0
)

In [17]:
df["high_risk_indicator"] = np.where(
    df["risk_score"].fillna(0) >= 70,
    1,
    0
)

df[[
    "risk_score",
    "high_risk_indicator"
]].head()

,risk_score,high_risk_indicator
0,10.0,0
1,35.0,0
2,28.0,0
3,33.0,0
4,53.0,0


In [18]:
df["risk_category"] = pd.cut(
    df["risk_score"],
    bins=[-1, 30, 60, 100],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

df[["risk_score", "risk_category"]].head()

,risk_score,risk_category
0,10.0,Low Risk
1,35.0,Medium Risk
2,28.0,Low Risk
3,33.0,Medium Risk
4,53.0,Medium Risk


In [19]:
indicator_columns = [
    "unusual_time_indicator",
    "high_amount_indicator",
    "amount_above_average_indicator",
    "high_frequency_indicator",
    "failed_attempt_indicator",
    "far_from_home_indicator",
    "new_device_indicator",
    "new_location_indicator",
    "international_indicator",
    "card_not_present_indicator",
    "high_risk_indicator"
]

df["total_fraud_indicators"] = df[indicator_columns].sum(axis=1)

df[indicator_columns + ["total_fraud_indicators"]].head()

,unusual_time_indicator,high_amount_indicator,amount_above_average_indicator,high_frequency_indicator,failed_attempt_indicator,far_from_home_indicator,new_device_indicator,new_location_indicator,international_indicator,card_not_present_indicator,high_risk_indicator,total_fraud_indicators
0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,1
2,0,0,0,0,0,0,0,0,1,0,0,1
3,0,0,0,0,0,0,0,0,0,1,0,1
4,1,0,0,0,0,0,0,0,1,1,0,3


In [20]:
df["suspicious_transaction"] = np.where(
    df["total_fraud_indicators"] >= 3,
    1,
    0
)

df[[
    "total_fraud_indicators",
    "suspicious_transaction",
    "is_fraud"
]].head()

,total_fraud_indicators,suspicious_transaction,is_fraud
0,0,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,3,1,0


In [21]:
def assign_fraud_risk(indicator_count):
    if indicator_count <= 1:
        return "Low"
    elif indicator_count <= 3:
        return "Medium"
    else:
        return "High"


df["engineered_fraud_risk"] = df[
    "total_fraud_indicators"
].apply(assign_fraud_risk)

df[[
    "total_fraud_indicators",
    "engineered_fraud_risk"
]].head()

,total_fraud_indicators,engineered_fraud_risk
0,0,Low
1,1,Low
2,1,Low
3,1,Low
4,3,Medium


In [22]:
print(
    df["suspicious_transaction"].value_counts()
)

suspicious_transaction
0    56133
1     1528
Name: count, dtype: int64


In [23]:
fraud_indicator_summary = (
    df.groupby("engineered_fraud_risk")["is_fraud"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

fraud_indicator_summary["mean"] *= 100

fraud_indicator_summary.rename(columns={
    "count": "total_transactions",
    "sum": "actual_fraud_transactions",
    "mean": "fraud_rate_percent"
}, inplace=True)

fraud_indicator_summary

,engineered_fraud_risk,total_transactions,actual_fraud_transactions,fraud_rate_percent
0,High,169,6,3.550296
1,Low,48340,779,1.611502
2,Medium,9152,213,2.327360


In [24]:
comparison_table = pd.crosstab(
    df["suspicious_transaction"],
    df["is_fraud"],
    rownames=["Suspicious Indicator"],
    colnames=["Actual Fraud"]
)

comparison_table

Actual Fraud,0,1
Suspicious Indicator,,
0,55175,958
1,1488,40


In [25]:
df.to_csv(
    "fraud_dataset_with_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!
